# without stimulus


In [1]:
# imports
import snntorch as snn
from snntorch import spikeplot as splt
from snntorch import spikegen

import torch
import torch.nn as nn

import numpy as np
import matplotlib.pyplot as plt

In [2]:
def leaky_integrate_neuron(U, time_step=1e-3, I=0, R=5e7, C=1e-10):
  tau = R*C
  U = U + (time_step/tau)*(-U + I*R)
  return U

In [4]:
num_steps = 100
U = 0.9
U_trace = []  # keeps a record of U for plotting

for step in range(num_steps):
  U_trace.append(U)
  U = leaky_integrate_neuron(U)  # solve next step of U

print(U_trace)


[0.9, 0.72, 0.576, 0.4608, 0.36863999999999997, 0.29491199999999995, 0.23592959999999996, 0.18874367999999997, 0.15099494399999996, 0.12079595519999997, 0.09663676415999997, 0.07730941132799998, 0.061847529062399986, 0.04947802324991999, 0.03958241859993599, 0.031665934879948794, 0.025332747903959035, 0.020266198323167228, 0.016212958658533783, 0.012970366926827025, 0.01037629354146162, 0.008301034833169296, 0.006640827866535437, 0.00531266229322835, 0.00425012983458268, 0.003400103867666144, 0.0027200830941329153, 0.0021760664753063324, 0.001740853180245066, 0.0013926825441960527, 0.0011141460353568421, 0.0008913168282854737, 0.0007130534626283789, 0.0005704427701027032, 0.00045635421608216254, 0.00036508337286573003, 0.000292066698292584, 0.00023365335863406722, 0.00018692268690725377, 0.000149538149525803, 0.0001196305196206424, 9.570441569651392e-05, 7.656353255721113e-05, 6.125082604576891e-05, 4.9000660836615124e-05, 3.92005286692921e-05, 3.1360422935433675e-05, 2.508833834834694

# with stimulus

In [32]:
time_step = 1e-3
R = 5
C = 1e-3
num_steps = 10

# leaky integrate and fire neuron, tau=5e-3
lif1 = snn.Lapicque(R=R, C=C, time_step=time_step)

# set up membrane
membrane = torch.ones(1) * 0.9 # U = 0.9 at t=0
cur_in = torch.zeros(num_steps, 1) # I = 0 for all t... where does the 1 come in?
spk_out = torch.zeros(1) # init output spike...
membrance_recorder = [membrane] # records value of membrane at every time step

In [33]:
for step in range(num_steps):
    spk_out, membrane = lif1(cur_in[step], membrane)
    membrance_recorder.append(membrane)

print("membrane recorder before stack")
print(len(membrance_recorder), len(membrance_recorder[0])) # [0], membrance_recorder[0][0][0])
membrance_recorder = torch.stack(membrance_recorder)
print("membrane recorder after stack")
print(membrance_recorder.size())

membrane recorder before stack
11 1
membrane recorder after stack
torch.Size([11, 1])


In [38]:
cur_in = torch.cat((torch.zeros(10, 1), torch.ones(190, 1)*0.1), 0)  # input current turns on at t=10

# Initialize membrane, output and recordings
membrane = torch.zeros(1)  # membrane potential of 0 at t=0
spk_out = torch.zeros(1)  # neuron needs somewhere to sequentially dump its output spikes
membrance_recorder = [membrane]


In [39]:
num_steps = 200
for step in range(num_steps):
    # for each step
    # get back whether there's been a spike (what do they do with this)
    # and record the membrane potential
    spk_out, membrane = lif1(cur_in[step], membrane)
    membrance_recorder.append(membrane)

membrance_recorder = torch.stack(membrance_recorder)
print(membrance_recorder)
print(f"The calculated value of input pulse [A] x resistance [Ω] is: {cur_in[11]*lif1.R} V")
print(f"The simulated value of steady-state membrane potential is: {membrance_recorder[200][0]} V")


tensor([[0.0000],
        [0.0000],
        [0.0000],
        [0.0000],
        [0.0000],
        [0.0000],
        [0.0000],
        [0.0000],
        [0.0000],
        [0.0000],
        [0.0000],
        [0.1000],
        [0.1800],
        [0.2440],
        [0.2952],
        [0.3362],
        [0.3689],
        [0.3951],
        [0.4161],
        [0.4329],
        [0.4463],
        [0.4571],
        [0.4656],
        [0.4725],
        [0.4780],
        [0.4824],
        [0.4859],
        [0.4887],
        [0.4910],
        [0.4928],
        [0.4942],
        [0.4954],
        [0.4963],
        [0.4970],
        [0.4976],
        [0.4981],
        [0.4985],
        [0.4988],
        [0.4990],
        [0.4992],
        [0.4994],
        [0.4995],
        [0.4996],
        [0.4997],
        [0.4997],
        [0.4998],
        [0.4998],
        [0.4999],
        [0.4999],
        [0.4999],
        [0.4999],
        [0.4999],
        [0.5000],
        [0.5000],
        [0.5000],
        [0

In [50]:
cur_in1 = torch.cat((torch.zeros(10, 1), torch.ones(20, 1)*(0.1), torch.zeros(170, 1)), 0)  # input turns on at t=10, off at t=30
mem = torch.zeros(1)
spk_out = torch.zeros(1)
mem_rec1 = [mem]
for step in range(num_steps):
    spk_out, mem = lif1(cur_in[step], mem)
    mem_rec1.append(mem)
mem_rec1 = torch.stack(mem_rec1)
# print(mem_rec1)
print(f"The calculated value of input pulse [A] x resistance [Ω] is: {cur_in[11]*lif1.R} V")
print(f"The simulated value of steady-state membrane potential is: {mem_rec1[200][0]} V")

# cur_in1

The calculated value of input pulse [A] x resistance [Ω] is: tensor([0.5000]) V
The simulated value of steady-state membrane potential is: 0.4999999403953552 V
